In [2]:
import os
import mysql.connector
from dotenv import load_dotenv

ModuleNotFoundError: No module named 'mysql'

In [21]:
load_dotenv()
def get_connection():
        conn = mysql.connector.connect(
            host=os.getenv("MYSQL_HOST"),
            user=os.getenv("MYSQL_USER"),
            password=os.getenv("MYSQL_PASSWORD"),
            database=os.getenv("MYSQL_DATABASE")
        )

        return conn

In [22]:
def get_city_summary(city: str): 
    conn = get_connection()
    cursor = conn.cursor(dictionary=True)
    query = """
    SELECT 
        City, 
        COUNT(*)                                       AS sold_count, 
        ROUND(AVG(ClosePrice), 0)                      AS avg_close_price, 
        ROUND(AVG(ClosePrice / NULLIF(LivingArea,0)),0) AS avg_price_per_sqft, 
        ROUND(AVG(DaysOnMarket), 1)                    AS avg_dom, 
        ROUND(AVG(ClosePrice / NULLIF(ListPrice,0)) * 100, 1) AS list_to_close_pct 
    FROM california_sold 
    WHERE PropertyType = 'Residential' 
        AND City = %s
        AND CloseDate >= DATE_SUB(CURDATE(), INTERVAL 12 MONTH) 
        AND LivingArea > 0 
    GROUP BY City 
    ORDER BY sold_count DESC 
    LIMIT 25; 
    """
    cursor.execute(query, (city,))
    result = cursor.fetchone()
    cursor.close()
    conn.close()
    return result

In [23]:
summary = get_city_summary("Irvine")

print(summary)
print(summary["avg_close_price"])
print(summary["avg_price_per_sqft"])
print(summary["avg_dom"])
print(summary["list_to_close_pct"])

{'City': 'Irvine', 'sold_count': 991, 'avg_close_price': 1741302.0, 'avg_price_per_sqft': 811.0, 'avg_dom': Decimal('44.4'), 'list_to_close_pct': 98.1}
1741302.0
811.0
44.4
98.1
